# 02g — InceptionV3 + CBAM Hierarchical (Branching Heads)

**Project:** UREP 32-0210-250078 | Crack Classification

**Architecture:** Single InceptionV3 backbone with 3 CBAM-equipped classification heads tapping `Mixed_5d` (35×35×288), `Mixed_6e` (17×17×768), and `Mixed_7c` (8×8×2048).

## Hierarchy

```
                Input image
                     │
         Stage 1: crack vs no_crack
                     │
              ┌──────┴──────┐
            crack          no_crack
              │
      Stage 2: single vs multi
              │
         ┌────┴────┐
       single    multi
         │
  Stage 3: 4-way subtype
  {debonding, flexural, shear, others}
```

## Notes

* Same 3-stage freeze/unfreeze schedule and same per-stage LR/epoch settings as `02b_training_cbam.ipynb`.
* Same augmentations (`get_train_transforms` reused unchanged).
* **Phase 2** uses the multi×3 sampler (`target_single = target_multi = 3·N_multi`).
* **Phase 3** uses a joint sampler that mixes ~25% no_crack into a balanced crack pool to keep head 1 from forgetting.
* Masked multi-task cross-entropy: each head only contributes loss for samples where its label is defined.
* Same 70/15/15 split as the flat baseline → directly comparable.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import json

import numpy as np
import torch

import config
from src.device import print_device_summary, get_device, set_seed
from src.evaluation import evaluate_predictions
from src.model_cbam_hierarchical import (
    InceptionV3CBAMHierarchical, freeze_backbone, unfreeze_segment_b, unfreeze_all,
)
from src.hierarchical import (
    STAGE1_CLASSES, STAGE2_CLASSES, STAGE3_CLASSES,
    get_hierarchical_dataloaders,
    train_hierarchical_model,
    evaluate_hierarchical_model,
    cascaded_to_orig,
    hierarchical_pr_f1, per_stage_confusion_matrices, error_attribution,
)

# Reproducibility
set_seed(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "cbam_hier")
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "logs"), exist_ok=True)

device_config = print_device_summary()
device = get_device()
STAGE_BATCH = device_config["batch_sizes"]
NUM_WORKERS = device_config["num_workers"]

print(f"\nModel: InceptionV3 + CBAM (hierarchical, 3 heads)")
print(f"Device: {device}")

## Build hierarchical model

In [ ]:
model = InceptionV3CBAMHierarchical().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Sanity-check the 3 head shapes on a dummy batch
with torch.no_grad():
    dummy = torch.zeros(2, 3, config.IMG_SIZE, config.IMG_SIZE, device=device)
    o1, o2, o3 = model(dummy)
    print(f"Head1 (stage1): {tuple(o1.shape)}  (expect (2, {len(STAGE1_CLASSES)}))")
    print(f"Head2 (stage2): {tuple(o2.shape)}  (expect (2, {len(STAGE2_CLASSES)}))")
    print(f"Head3 (stage3): {tuple(o3.shape)}  (expect (2, {len(STAGE3_CLASSES)}))")

## Compute per-stage class weights

Used inside the masked cross-entropy losses for each head.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from src.hierarchical import HierarchicalCrackDataset, STAGE1_TO_IDX, STAGE2_TO_IDX, STAGE3_TO_IDX

_train_ds = HierarchicalCrackDataset(config.SPLIT_DIR, "train", transform=None)

def _cw(arr, n_classes):
    arr = np.asarray(arr)
    if len(arr) == 0:
        return torch.ones(n_classes, dtype=torch.float32)
    classes = np.arange(n_classes)
    present = np.unique(arr)
    w = compute_class_weight("balanced", classes=present, y=arr)
    out = np.ones(n_classes, dtype=np.float32)
    for c, wc in zip(present, w):
        out[c] = wc
    return torch.tensor(out, dtype=torch.float32)

cw_s1 = _cw(_train_ds.s1, len(STAGE1_CLASSES))
cw_s2 = _cw(_train_ds.s2[_train_ds.m2 > 0], len(STAGE2_CLASSES))
cw_s3 = _cw(_train_ds.s3[_train_ds.m3 > 0], len(STAGE3_CLASSES))

class_weights = {"stage1": cw_s1, "stage2": cw_s2, "stage3": cw_s3}
print(f"Stage1 weights ({STAGE1_CLASSES}): {cw_s1.tolist()}")
print(f"Stage2 weights ({STAGE2_CLASSES}): {cw_s2.tolist()}")
print(f"Stage3 weights ({STAGE3_CLASSES}): {cw_s3.tolist()}")
del _train_ds

## Phase 1 — Feature extraction (Head 1 only)

Backbone frozen, sampler balances stage-1 classes, only head 1 contributes loss.

In [ ]:
train_loader, val_loader, _ = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[1],
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="stage1",
)

freeze_backbone(model)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=config.STAGE1_LR,
)

history1 = train_hierarchical_model(
    model, train_loader, val_loader, optimizer, device,
    epochs=config.STAGE1_EPOCHS, output_dir=OUTPUT_DIR, stage=1,
    loss_weights=(1.0, 0.0, 0.0),
    class_weights=class_weights,
    model_name="cbam_hier",
)

## Phase 2 — Partial fine-tuning (Heads 1 + 2)

Unfreeze segments b+c. Use the **multi×3** sampler. Loss weights `(0.3, 0.7, 0.0)`.
Stage-1 head still receives a small loss share so it doesn't drift on the crack-heavy batches.

In [ ]:
train_loader, val_loader, _ = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[2],
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="stage2",
)

unfreeze_segment_b(model)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=config.STAGE2_LR,
)

history2 = train_hierarchical_model(
    model, train_loader, val_loader, optimizer, device,
    epochs=config.STAGE2_EPOCHS, output_dir=OUTPUT_DIR, stage=2,
    loss_weights=(0.3, 0.7, 0.0),
    class_weights=class_weights,
    model_name="cbam_hier",
)

## Phase 3 — Full fine-tuning (all 3 heads)

Unfreeze everything. Joint sampler mixes 25% no_crack into a stage-3-balanced crack pool.
Loss weights `(0.2, 0.3, 0.5)` — most weight on the hardest task.

In [ ]:
train_loader, val_loader, test_loader = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[3],
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="joint",
    no_crack_mix=0.25,
)

unfreeze_all(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.STAGE3_LR)

history3 = train_hierarchical_model(
    model, train_loader, val_loader, optimizer, device,
    epochs=config.STAGE3_EPOCHS, output_dir=OUTPUT_DIR, stage=3,
    loss_weights=(0.2, 0.3, 0.5),
    class_weights=class_weights,
    model_name="cbam_hier",
)

## Training curves

Plots per-head loss and accuracy across all 3 phases.

In [ ]:
import matplotlib.pyplot as plt

histories = [history1, history2, history3]
phase_names = ["Phase 1 — Feature Extraction", "Phase 2 — Partial FT", "Phase 3 — Full FT"]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
metric_pairs = [
    ("loss_s1", "val_loss_s1", "Stage 1 loss"),
    ("loss_s2", "val_loss_s2", "Stage 2 loss"),
    ("loss_s3", "val_loss_s3", "Stage 3 loss"),
    ("acc_s1",  "val_acc_s1",  "Stage 1 accuracy"),
    ("acc_s2",  "val_acc_s2",  "Stage 2 accuracy"),
    ("acc_s3",  "val_acc_s3",  "Stage 3 accuracy"),
]

for ax, (tk, vk, title) in zip(axes.flat, metric_pairs):
    offset = 0
    for h, name, color in zip(histories, phase_names, colors):
        n = len(h[tk])
        xs = range(offset, offset + n)
        ax.plot(xs, h[tk], color=color, linestyle="-", label=f"{name} (train)")
        ax.plot(xs, h[vk], color=color, linestyle="--", label=f"{name} (val)")
        if offset > 0:
            ax.axvline(x=offset, color="gray", linestyle=":", alpha=0.5)
        offset += n
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "training_history_cbam_hier.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## Save final model

In [ ]:
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "models", "best_model.pt"))
print("Saved final hierarchical CBAM model.")

## Cascaded test-set evaluation

In [ ]:
_, _, test_loader = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=STAGE_BATCH[1],
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="stage1",   # only train_loader uses the sampler; test_loader is sequential
)

results = evaluate_hierarchical_model(model, test_loader, device, t1=0.5, t2=0.5)

y_true_idx = np.array([config.CLASS_NAMES.index(c) for c in results["y_true_flat"]])
y_pred_idx = np.array([config.CLASS_NAMES.index(c) for c in results["y_pred_flat"]])
metrics = evaluate_predictions(
    y_true_idx, y_pred_idx,
    output_dir=OUTPUT_DIR, model_name="cbam_hier",
)

## Per-stage confusion matrices, hierarchical metrics, error attribution

In [ ]:
import seaborn as sns

stage_cms = per_stage_confusion_matrices(results["y_true_paths"], results["y_pred_paths"])
h_metrics = hierarchical_pr_f1(results["y_true_paths"], results["y_pred_paths"])
err_attr  = error_attribution(results["y_true_paths"], results["y_pred_paths"])

print("Hierarchical precision/recall/F1:")
for k, v in h_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nError attribution:")
print(f"  Total:        {err_attr['total']}")
print(f"  Correct:      {err_attr['correct']}")
print(f"  Stage1 errs:  {err_attr['errors_by_stage']['stage1']}")
print(f"  Stage2 errs:  {err_attr['errors_by_stage']['stage2']}")
print(f"  Stage3 errs:  {err_attr['errors_by_stage']['stage3']}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key in zip(axes, ["stage1", "stage2", "stage3"]):
    if key not in stage_cms:
        ax.set_visible(False); continue
    info = stage_cms[key]
    sns.heatmap(info["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=info["classes"], yticklabels=info["classes"], ax=ax)
    ax.set_title(f"{key}  ({info['cm'].sum()} samples)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "per_stage_confusion_matrices.png"),
            dpi=150, bbox_inches="tight")
plt.show()

with open(os.path.join(OUTPUT_DIR, "hierarchical_metrics.json"), "w") as f:
    json.dump({
        "hierarchical": h_metrics,
        "error_attribution": err_attr,
        "flat": {
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
        },
    }, f, indent=2)
print(f"\nSaved hierarchical metrics to {OUTPUT_DIR}/hierarchical_metrics.json")